# **Dataset Merging**

1. Dataset1: dataset_gizi_polaku.csv
2. Dataset2: dataset_gizi_nutrition.csv

In [ ]:
import pandas as pd

In [8]:
# 1. MEMUAT DATASET
print("--- MEMUAT DATASET ---")
file_dataset1 = 'data/dataset_gizi_nutrition.csv'
file_dataset2 = 'data/dataset_gizi_polaku.csv'

try:
    df1 = pd.read_csv(file_dataset1)
    print(f"Dataset 1 ({file_dataset1}) dimuat dengan {df1.shape[0]} baris dan {df1.shape[1]} kolom.")
except FileNotFoundError:
    print(f"Error: {file_dataset1} tidak ditemukan.")
    exit()

try:
    df2 = pd.read_csv(file_dataset2)
    print(f"Dataset 2 ({file_dataset2}) dimuat dengan {df2.shape[0]} baris dan {df2.shape[1]} kolom.")
except FileNotFoundError:
    print(f"Error: {file_dataset2} tidak ditemukan.")
    exit()

--- MEMUAT DATASET ---
Dataset 1 (data/dataset_gizi_nutrition.csv) dimuat dengan 1185 baris dan 8 kolom.
Dataset 2 (data/dataset_gizi_polaku.csv) dimuat dengan 2616 baris dan 6 kolom.


In [9]:
# 2. STANDARISASI DAN PEMILIHAN KOLOM
print("\n--- STANDARISASI KOLOM ---")

# Kolom target yang diinginkan untuk hasil akhir
target_columns = ['Nama Makanan', 'Ukuran Porsi', 'Kalori (kkal)', 'Karbohidrat (g)', 'Lemak (g)', 'Protein (g)']

# Dataset 1 sudah sesuai dengan target columns (hanya memastikan urutan)
df1_final = df1[target_columns].copy()

# Dataset 2 perlu perbaikan nama kolom
# Mapping nama kolom lama ke nama kolom target
rename_mapping = {
    'Ukutan Porsi': 'Ukuran Porsi',  # Memperbaiki typo
    'Kalori (Kal)': 'Kalori (kkal)', # Menyelaraskan satuan kalori
    'Lemak': 'Lemak (g)'             # Menyelaraskan satuan lemak
}

# Ganti nama kolom di dataset 2
df2_renamed = df2.rename(columns=rename_mapping)

# Pilih hanya kolom target (abaikan Kategori dan Glycemic Risk)
df2_final = df2_renamed[target_columns].copy()

print("Kolom berhasil diselaraskan.")


--- STANDARISASI KOLOM ---
Kolom berhasil diselaraskan.


In [10]:
# 3. MENGGABUNGKAN DATASET (MERGE/CONCAT)
print("\n--- MENGGABUNGKAN DATA ---")

# Gabungkan secara vertikal (tumpuk)
df_merged = pd.concat([df1_final, df2_final], ignore_index=True)
print(f"Total baris setelah penggabungan (sebelum hapus duplikat): {df_merged.shape[0]}")


--- MENGGABUNGKAN DATA ---
Total baris setelah penggabungan (sebelum hapus duplikat): 3801


In [11]:
# 4. PENANGANAN DUPLIKAT
print("\n--- MEMBERSIHKAN DUPLIKAT ---")

# Untuk memastikan deteksi duplikat akurat (misal "Nasi" vs "nasi"), kita
# buat kolom sementara berisi nama makanan dengan huruf kecil (lowercase).
df_merged['Nama_Lower'] = df_merged['Nama Makanan'].astype(str).str.lower().str.strip()

# Menghapus duplikat berdasarkan kolom 'Nama_Lower'.
# keep='first' berarti kita menyimpan kemunculan pertama (biasanya dari Dataset 1).
df_merged = df_merged.drop_duplicates(subset=['Nama_Lower'], keep='first')

# Hapus kolom sementara 'Nama_Lower' karena sudah tidak dipakai
df_merged = df_merged.drop(columns=['Nama_Lower'])

print(f"Total baris setelah menghapus makanan duplikat: {df_merged.shape[0]}")

# Opsional: Rapikan kembali penulisan nama makanan menjadi Title Case (Huruf Kapital di awal kata)
df_merged['Nama Makanan'] = df_merged['Nama Makanan'].str.title()


--- MEMBERSIHKAN DUPLIKAT ---
Total baris setelah menghapus makanan duplikat: 3210


In [12]:
# 5. MENYIMPAN HASIL
print("\n--- PROSES SELESAI ---")
print("\n5 Baris Pertama Dataset Final:")
print(df_merged.head())

# Menyimpan ke file CSV baru
output_file = 'dataset_gizi_merged_final.csv'
df_merged.to_csv(output_file, index=False)

print(f"\nDataset berhasil digabungkan dan disimpan sebagai '{output_file}'")


--- PROSES SELESAI ---

5 Baris Pertama Dataset Final:
           Nama Makanan Ukuran Porsi  Kalori (kkal)  Karbohidrat (g)  \
0                  Nasi          100          180.0             39.8   
1              Nasi Tim          100          120.0             26.0   
2   Bihun Goreng Instan          100          381.0             80.3   
3  Bihun Jagung, Mentah          100          354.0             87.4   
4   Jagung Nasi, Mentah          100          357.0             79.5   

   Lemak (g)  Protein (g)  
0        0.3          3.0  
1        0.4          2.4  
2        3.9          6.1  
3        0.3          0.5  
4        0.5          8.8  

Dataset berhasil digabungkan dan disimpan sebagai 'dataset_gizi_merged_final.csv'
